In [3]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pdfplumber pandas

import os, re
import pandas as pd
import pdfplumber

IN_DIR = "/content/drive/MyDrive/Math Olympiad Competition/svsu_pdfs"
OUT_CSV_DIR = "/content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV"
os.makedirs(OUT_CSV_DIR, exist_ok=True)

print("PDF folder contains:", len([f for f in os.listdir(IN_DIR) if f.lower().endswith(".pdf")]), "PDFs")

def extract_text_from_pdf(pdf_path: str) -> str:
    pages = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                # layout=True sometimes helps preserve numbering/line breaks
                pages.append(page.extract_text(layout=True) or "")
        return "\n".join(pages)
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
        return ""

def split_problems(full_text: str):
    """
    Split on problem numbers that look like:
      1.   or  1)   (allow spaces like 1 ) or 1 .)
    Must appear at start of a line.
    """
    pat = r"(?m)^\s*(\d+)\s*[\.\)]\s+"
    matches = list(re.finditer(pat, full_text))
    chunks = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(full_text)
        chunks.append(full_text[start:end].strip())
    return chunks

def problem_only(chunk: str) -> str:
    """
    Keep only the problem statement; remove everything from Solution / Solutions onward.
    Handles:
      Solution:
      Solutions:
      Solution (E):
      SOLUTION :
    """
    return re.split(r"(?i)\bsolutions?\b\s*(?:\([^)]+\))?\s*:\s*", chunk, maxsplit=1)[0].strip()

def normalize_whitespace(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def file_id_prefix(pdf_filename: str) -> str:
    """
    Make stable ID prefix from the filename.
    Example: 2003level2_solutionsFinal.pdf -> svsu_2003level2
    """
    base = os.path.splitext(pdf_filename)[0].lower()
    base = re.sub(r"[^a-z0-9]+", "_", base).strip("_")
    return f"svsu_{base}"

pdf_files = sorted([f for f in os.listdir(IN_DIR) if f.lower().endswith(".pdf")])

summary = []
for fn in pdf_files:
    pdf_path = os.path.join(IN_DIR, fn)
    text = extract_text_from_pdf(pdf_path)

    if not text.strip():
        print("SKIP empty text:", fn)
        summary.append((fn, 0, "empty text"))
        continue

    chunks = split_problems(text)

    # If no chunks found, log it so we can adjust for that file format
    if len(chunks) == 0:
        print("NO MATCHES (check numbering format):", fn)
        summary.append((fn, 0, "no matches"))
        continue

    prefix = file_id_prefix(fn)
    rows = []
    for idx, ch in enumerate(chunks, start=1):
        ptxt = problem_only(ch)
        # remove leading "n." or "n)" from the chunk
        ptxt = re.sub(r"^\s*\d+\s*[\.\)]\s*", "", ptxt).strip()
        ptxt = normalize_whitespace(ptxt)
        if ptxt:
            rows.append({"id": f"{prefix}_{idx:02d}", "problem": ptxt})

    df = pd.DataFrame(rows, columns=["id", "problem"])
    out_csv = os.path.join(OUT_CSV_DIR, os.path.splitext(fn)[0] + "_problems.csv")
    df.to_csv(out_csv, index=False)

    print(f"OK  {fn}: {len(df)} problems -> {out_csv}")
    summary.append((fn, len(df), out_csv))

print("\nDone.")
print("Files with 0 extracted problems:")
for fn, n, note in summary:
    if n == 0:
        print(" -", fn, "=>", note)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PDF folder contains: 97 PDFs
NO MATCHES (check numbering format): 2000Solutions_letter.pdf
OK  2001Solutions_letter.pdf: 50 problems -> /content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV/2001Solutions_letter_problems.csv
OK  2003Level1_Solutions.pdf: 25 problems -> /content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV/2003Level1_Solutions_problems.csv
NO MATCHES (check numbering format): 2003Solutions_letter.pdf
OK  2003level2_solutionsFinal.pdf: 26 problems -> /content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV/2003level2_solutionsFinal_problems.csv
OK  2004_Level_1_Final_Solutions.pdf: 25 problems -> /content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV/2004_Level_1_Final_Solutions_problems.csv
OK  2004_Level_2_Final_solutions.pdf: 26 problems -> /content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV/2004_Level_2_

In [6]:
import os, glob, re
import pandas as pd

OUT_CSV_DIR = "/content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV"
COMBINED_CSV = os.path.join(OUT_CSV_DIR, "svsu_all_problems_combined.csv")

def base26_suffix(n: int, width: int = 3) -> str:
    chars = []
    for _ in range(width):
        n, rem = divmod(n, 26)
        chars.append(chr(ord('a') + rem))
    return "".join(reversed(chars))

def make_id(i: int) -> str:
    return f"{i:03d}{base26_suffix(i, 3)}"

# ---- NEW: cleaning helpers ----
UNICODE_MAP = {
    "−": "-", "–": "-", "—": "-",
    "⋅": r"\cdot", "·": r"\cdot",
    "×": r"\times", "÷": r"\div",
    "≤": r"\le", "≥": r"\ge", "≠": r"\neq",
    "½": r"\frac{1}{2}", "¼": r"\frac{1}{4}", "¾": r"\frac{3}{4}",
    "π": r"\pi", "θ": r"\theta",
    "√": r"\sqrt",
}

def strip_choices(s: str) -> str:
    """Cut off multiple-choice options starting at first (a)/a)/a. marker."""
    if s is None:
        return ""
    s = str(s)
    patterns = [
        r"\s\([a-e]\)\s",  # (a)
        r"\s[a-e]\)\s",    # a)
        r"\s[a-e]\.\s",    # a.
    ]
    starts = []
    for pat in patterns:
        m = re.search(pat, s, flags=re.IGNORECASE)
        if m:
            starts.append(m.start())
    if starts:
        s = s[:min(starts)]
    return s.strip()

def replace_unicode(s: str) -> str:
    for k, v in UNICODE_MAP.items():
        s = s.replace(k, v)
    return s

def convert_powers(s: str) -> str:
    # x2 -> x^2 (only when adjacent)
    return re.sub(r"([A-Za-z])(\d{1,2})(?!\d)", r"\1^\2", s)

MATH_TRIGGER_RE = re.compile(r"(\\cdot|\\times|\\div|\\frac|\\sqrt|\\le|\\ge|\\neq|\^|[=<>]|\|)")

def wrap_math_segments(text: str) -> str:
    """Best-effort wrap mathy token runs in $...$ without touching existing $...$."""
    parts = text.split("$")
    out_parts = []
    for i, part in enumerate(parts):
        if i % 2 == 1:
            out_parts.append(part)  # already math
            continue

        def repl(m):
            seg = m.group(0)
            if not MATH_TRIGGER_RE.search(seg):
                return seg
            return f"${seg}$"

        part = re.sub(r"[A-Za-z0-9\\\^\{\}_\(\)\[\]\+\-\*/=<>\|,\.]+", repl, part)
        out_parts.append(part)

    return "$".join(out_parts)

def clean_problem(s: str) -> str:
    s = strip_choices(s)
    s = replace_unicode(s)
    s = convert_powers(s)

    # \sqrt 5 -> \sqrt{5}
    s = re.sub(r"\\sqrt\s*([A-Za-z0-9]+)", r"\\sqrt{\1}", s)

    # fix common join like "6is" -> "6 is"
    s = re.sub(r"(\d)(is)\b", r"\1 \2", s)

    # normalize whitespace
    s = re.sub(r"\s+", " ", str(s)).strip()

    # best-effort math wrapping
    s = wrap_math_segments(s)

    # cleanup around $
    s = re.sub(r"\$\s+", "$", s)
    s = re.sub(r"\s+\$", "$", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ---- UPDATED: stricter junk filter ----
def is_bad_problem_text(s: str) -> bool:
    if s is None:
        return True
    s = str(s).strip()
    if s == "":
        return True

    # answer-key letter
    if re.fullmatch(r"\(?\s*[A-Ea-e]\s*\)?\.?", s):
        return True

    # too short after cleaning
    if len(s) < 15:
        return True

    # must contain some letters (helps drop weird numeric-only scraps)
    if not re.search(r"[A-Za-z]", s):
        return True

    # mostly non-alphanumeric
    alnum = sum(ch.isalnum() for ch in s)
    if alnum < 8:
        return True

    return False

# ---- COMBINE ----
csv_files = sorted(glob.glob(os.path.join(OUT_CSV_DIR, "*_problems.csv")))
print("Found per-PDF CSVs:", len(csv_files))

dfs = []
for path in csv_files:
    df = pd.read_csv(path)
    if "problem" not in df.columns:
        continue

    df = df[["problem"]].copy()
    df["problem"] = df["problem"].astype(str).map(clean_problem)
    df = df[~df["problem"].apply(is_bad_problem_text)]
    dfs.append(df)

if not dfs:
    raise ValueError("No usable rows found in *_problems.csv files.")

all_df = pd.concat(dfs, ignore_index=True).reset_index(drop=True)

# Assign ids AFTER filtering/cleaning so they stay sequential
all_df.insert(0, "id", [make_id(i) for i in range(len(all_df))])

# Only keep these two columns
all_df = all_df[["id", "problem"]]

all_df.to_csv(COMBINED_CSV, index=False)
print("Wrote:", COMBINED_CSV)
print("Total kept problems:", len(all_df))
all_df.head(20)


Found per-PDF CSVs: 77
Wrote: /content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV/svsu_all_problems_combined.csv
Total kept problems: 1782


,id,problem
0,000aaa,Which of these numbers is largest?
1,001aab,Last year a bicycle cost$160 and a cycling hel...
2,002aac,A vacuum pump removes$\frac{1}{2}$of the air i...
3,003aad,"The ratio of wto x is 4:3, of y to zis 3:2, an..."
4,004aae,Find the difference of and . x+1$x^2$-1 -2
5,005aaf,The number of real solutions of the equation$|...
6,006aag,The sum of the solutions to$x^2$$-x=6$is
7,007aah,"If$f(x)=1-x^2,$find a constant cso that$=c(2a+..."
8,008aai,"If$m>0$and the points (m,3) and (1,m) lie on a..."
9,009aaj,For how many integers nbetween 1 and 100 does$...
